What is LSTM? 

LSTM: Long Short Term Memory is a neural network architecture that allows RNNs to retain information over long sequences. It has a gradient highway where the previous layer's output is passed through and added to the processed input token. The cell state allows information to persist. There are 3 gating mechanisms thea control the flow of information. A forget gate which decides whther to discard the information from the previous cell state. Input gate that determines which new information to store in the cell state. Output gate that decides what part of the cell state to output as the hidden state/, 


Gates that matter for volatility forecasting 
Forget gate is important as it allows us to just focus on the volatility numbers that matter such as sudden drops or sudden increases in prices. 
input gate is also important as it determines whether the current input is important

We need to feed into LSTM log_returns as we need continuous data about the exact price change of a feature. We also need to add volatility as it captures the recent turbulence and variability the market is in right now. Using both it can give us a context of how th emarket is right now. Raw prices would not work as othe values arent normalized.

The tradeoffs of using a longer window would be that we have more data about how the feature moves along time. Thus, we can determine if an abrupt change in price or high volatility is common or is uncommon. Using shorter windows allows us to determine the best case scenario at the moment like a greedy algorithm. I think the LSTm for bitcoin should lookback around 30 days. 

In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm



In [4]:
def create_sequences(df: pd.DataFrame, window_size: int):
    X_data = df[['log_returns','volatility']].values
    y_data = df['volatility'].values
    X = [X_data[i:i+window_size] for i in range(len(df)-window_size)]
    y = [y_data[i+window_size] for i in range(len(df)-window_size)]
    
    return np.array(X), np.array(y)

In [7]:
# load csv 
btc_df = pd.read_csv('../data/btc_data.csv', index_col = 'Date', parse_dates = True)

train = btc_df.loc["2018-01-31":"2022-12-31"]
val = btc_df.loc["2023-01-01":"2023-12-31"]
test = btc_df.loc["2024-01-01":"2024-12-31"]

print(btc_df.index)

DatetimeIndex(['2018-01-31', '2018-02-01', '2018-02-02', '2018-02-03',
               '2018-02-04', '2018-02-05', '2018-02-06', '2018-02-07',
               '2018-02-08', '2018-02-09',
               ...
               '2024-12-21', '2024-12-22', '2024-12-23', '2024-12-24',
               '2024-12-25', '2024-12-26', '2024-12-27', '2024-12-28',
               '2024-12-29', '2024-12-30'],
              dtype='datetime64[us]', name='Date', length=2526, freq=None)


In [9]:
#Creating train, val, and test sets 

X_train, y_train = create_sequences(train, 30)
X_test, y_test = create_sequences(test, 30)
X_val, y_val = create_sequences(val, 30)

print(f"X_train shape: {X_train.shape}\ny_train shape: {y_train.shape}")

X_train shape: (1766, 30, 2)
y_train shape: (1766,)
